# Clase 10 — Outliers, Distribución Normal y Correlación

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador | UDLA

---

## Objetivos de hoy

1. **Outliers a fondo**: buenos vs malos, detección (IQR, Z-score), tratamiento y criterio de negocio
2. **Distribución Normal**: qué es, sus parámetros, por qué importa, cómo verificar, cómo simularla
3. **Correlación**: Pearson vs Spearman, heatmap, relaciones no lineales

**Dataset**: [Wine Quality (UCI)](https://archive.ics.uci.edu/ml/datasets/Wine+Quality) — datos reales de control de calidad de vino tinto.

## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

# Estilo de gráficos
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 5)

print("Librerías cargadas ✓")

## 1. Cargar el dataset Wine Quality

In [ ]:
url = ("https://archive.ics.uci.edu/ml/"
       "machine-learning-databases/wine-quality/"
       "winequality-red.csv")
df = pd.read_csv(url, sep=";")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

### ¿Qué significan las columnas?

| Variable | Descripción |
|---|---|
| fixed acidity | Ácidos no volátiles (tartárico) |
| volatile acidity | Ácidos volátiles (acético — en exceso = vinagre) |
| citric acid | Ácido cítrico (frescura) |
| residual sugar | Azúcar tras fermentación (g/L) |
| chlorides | Sal (cloruro de sodio, g/L) |
| free sulfur dioxide | SO₂ libre (antimicrobiano) |
| total sulfur dioxide | SO₂ total |
| density | Densidad (g/cm³) |
| pH | Acidez (0–14) |
| sulphates | Sulfatos (antimicrobiano) |
| alcohol | Grado alcohólico (% vol) |
| quality | Nota de catadores (0–10) |

---

# PARTE 1: Outliers a fondo

Un outlier es un valor que se aleja **significativamente** del resto. Pero "significativamente" depende del **contexto** y del **método**.

### Outliers "buenos" vs "malos"

| Tipo | Ejemplo | ¿Qué hacer? |
|---|---|---|
| **Bueno** (información) | Fraude, defecto de calidad, vino excepcional | **Investigar y mantener** |
| **Malo** (ruido) | Error de sensor, typo en captura, muestra contaminada | **Corregir o eliminar** |

> **La diferencia entre un outlier bueno y uno malo NO la determina el algoritmo — la determina el conocimiento del dominio.**

## 1.1 Visualizar la distribución de cada variable

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))

for ax, col in zip(axes.flat, df.columns):
    sns.histplot(df[col], kde=True, ax=ax, color="#C82B40")
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("")

# Apagar el último eje sobrante
axes.flat[-1].set_visible(False)

plt.suptitle("Distribución de cada variable", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Observaciones:**
- `residual sugar` y `chlorides` tienen colas largas a la derecha → candidatas a outliers.
- `pH` y `density` se ven bastante simétricas → ¿normales?
- `quality` es discreta (valores enteros de 3 a 8).

## 1.2 Detección con el método IQR

In [ ]:
def detectar_outliers_iqr(serie, factor=1.5):
    """Retorna una máscara booleana: True = outlier."""
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (serie < lower) | (serie > upper), lower, upper

In [ ]:
# Aplicar a residual sugar
col = "residual sugar"
mask, lower, upper = detectar_outliers_iqr(df[col])

print(f"Variable: {col}")
print(f"Límites IQR: [{lower:.2f}, {upper:.2f}]")
print(f"Outliers: {mask.sum()} de {len(df)} ({mask.mean()*100:.1f}%)")
print(f"\nEstadísticas de los outliers:")
df.loc[mask, col].describe()

In [ ]:
# Contar outliers en TODAS las columnas numéricas
resumen_outliers = []

for col in df.select_dtypes(include="number").columns:
    mask, lower, upper = detectar_outliers_iqr(df[col])
    resumen_outliers.append({
        "variable": col,
        "n_outliers": mask.sum(),
        "pct_outliers": round(mask.mean() * 100, 1),
        "limite_inf": round(lower, 3),
        "limite_sup": round(upper, 3),
    })

df_outliers = pd.DataFrame(resumen_outliers).sort_values("n_outliers", ascending=False)
df_outliers

## 1.3 Detección con Z-Score

### ¿Qué es el Z-Score?

El Z-score responde una pregunta simple: **"¿A cuántas desviaciones estándar está este dato del centro?"**

$$z = \frac{x - \mu}{\sigma}$$

| Símbolo | Nombre | Significado |
|:---:|---|---|
| $x$ | **El dato** | El valor individual que estamos evaluando |
| $\mu$ | **Media** ($\bar{x}$) | El "centro" de todos los datos |
| $\sigma$ | **Desviación estándar** | La "unidad de dispersión" — qué tan lejos se esparcen los datos típicamente alrededor de $\mu$ |
| $x - \mu$ | **Desviación** | Qué tan lejos está $x$ del centro, en las unidades originales (g/L, pH, etc.) |
| $z$ | **Z-score** | Esa misma distancia, pero expresada en **unidades de σ**. Es un número **sin unidades**, lo que permite comparar variables de escalas distintas |

### Interpretación:
- **|z| ≈ 0**: El dato está justo en el centro. Completamente normal.
- **|z| = 1**: A 1 desviación del centro. **Normal** — el 68% de los datos están aquí.
- **|z| = 2**: A 2 desviaciones. **Poco común** — solo el 5% están más lejos.
- **|z| ≥ 3**: A 3+ desviaciones. **Muy raro** — solo el 0.3% están más lejos → **candidato a outlier**.

In [ ]:
# Z-score paso a paso para entenderlo
col = "pH"
mu = df[col].mean()
sigma = df[col].std()

print(f"pH: media (μ) = {mu:.2f}, desv. estándar (σ) = {sigma:.2f}")
print(f"\nEjemplo numérico paso a paso:")
print(f"  Un vino tiene pH = 3.90")
print(f"  Paso 1 — Desviación: x - μ = 3.90 - {mu:.2f} = {3.90 - mu:.2f}")
print(f"  Paso 2 ��� Z-score:    z = {3.90 - mu:.2f} / {sigma:.2f} = {(3.90 - mu)/sigma:.2f}")
print(f"  Paso 3 — |z| = {abs((3.90 - mu)/sigma):.2f} > 3 → ¡Es outlier!")

print(f"\n{'pH':>8s} {'x - μ':>8s} {'z-score':>8s}   Interpretación")
print("-" * 55)

for valor in [3.20, 3.31, 3.50, 3.90, 2.74]:
    desviacion = valor - mu
    z = desviacion / sigma
    if abs(z) > 3:
        interp = "⚠️ OUTLIER (|z| > 3)"
    elif abs(z) > 2:
        interp = "Poco común"
    else:
        interp = "Normal"
    print(f"{valor:>8.2f} {desviacion:>8.2f} {z:>8.2f}   {interp}")

In [ ]:
# Ahora con scipy (la forma rápida para todos los datos)
col = "pH"
z_scores = stats.zscore(df[col])  # calcula z para TODOS los datos de una vez

# Ver los primeros 5
print("Primeros 5 z-scores:")
for i in range(5):
    print(f"  pH={df[col].iloc[i]:.2f}  →  z={z_scores[i]:+.2f}")

# Detectar outliers
threshold = 3
outliers_z = df[np.abs(z_scores) > threshold]
print(f"\nOutliers con |z| > {threshold}: {len(outliers_z)}")
print(f"Valores de pH extremos: {outliers_z[col].values}")

In [ ]:
# Comparar IQR vs Z-score en chlorides
col = "chlorides"

mask_iqr, _, _ = detectar_outliers_iqr(df[col])
mask_z = np.abs(stats.zscore(df[col])) > 3

print(f"Outliers en '{col}':")
print(f"  Método IQR:     {mask_iqr.sum()}")
print(f"  Método Z-score: {mask_z.sum()}")
print(f"  En común:       {(mask_iqr & mask_z).sum()}")
print(f"  Solo IQR:       {(mask_iqr & ~mask_z).sum()}")
print(f"  Solo Z-score:   {(~mask_iqr & mask_z).sum()}")

**Observación:** IQR suele detectar más outliers que Z-score. Esto es porque el Z-score se basa en media y std, que ya están **influenciados por los propios outliers** (la media se "jala" hacia ellos). IQR usa cuartiles, que son más robustos.

### ¿Qué es una heurística?

Tanto el factor 1.5 del IQR como el umbral de 3 del Z-score son **heurísticas**:

> **Heurística**: una regla práctica que funciona "suficientemente bien" en la mayoría de los casos, pero **no garantiza** el resultado correcto siempre. Es un atajo útil, no una verdad matemática.

Ejemplos:
- "Si |z| > 3, es outlier" → funciona bien con datos normales, pero no con datos sesgados
- "Si Q < Q1 - 1.5 × IQR, es outlier" → el 1.5 es una **convención** (propuesta por John Tukey en 1977), no una ley física
- "Si |skew| > 1, usar mediana" → buen criterio general, pero hay excepciones

**¿Por qué importa saber esto?** Para no aplicar reglas ciegamente. La heurística te da un **punto de partida**; el **criterio de negocio** te da la decisión final. *"El algoritmo me dijo que es outlier"* no es justificación suficiente para eliminar un dato.

## 1.4 Visualizar outliers

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cols = ["residual sugar", "chlorides", "volatile acidity",
        "sulphates", "total sulfur dioxide", "alcohol"]

for ax, col in zip(axes.flat, cols):
    sns.boxplot(y=df[col], ax=ax, color="#C82B40", width=0.4)
    sns.stripplot(y=df[col], ax=ax, color="#2D2D2D", alpha=0.15, size=2)
    ax.set_title(col, fontsize=11, fontweight="bold")

plt.suptitle("Boxplot + Stripplot: Outliers por variable", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 1.5 ¿Los outliers son vinos buenos o malos?

Antes de eliminar, **investiguemos**. ¿Qué calidad tienen los vinos con valores extremos?

In [ ]:
# Outliers de chlorides: ¿qué calidad tienen?
col = "chlorides"
mask, _, _ = detectar_outliers_iqr(df[col])

print("Calidad de vinos CON outliers en chlorides:")
print(df.loc[mask, "quality"].describe())
print(f"\nCalidad de vinos SIN outliers en chlorides:")
print(df.loc[~mask, "quality"].describe())

In [ ]:
# Visualizar la diferencia
fig, ax = plt.subplots(figsize=(8, 4))
df["es_outlier_chlorides"] = mask.map({True: "Outlier", False: "Normal"})

sns.boxplot(x="es_outlier_chlorides", y="quality", data=df,
            palette={"Normal": "#2563EB", "Outlier": "#C82B40"}, ax=ax)
ax.set_title("¿Los vinos con chlorides extremos son peores?", fontsize=12)
plt.show()

# Limpiar la columna temporal
df.drop(columns="es_outlier_chlorides", inplace=True)

## 1.6 ¿Qué hacer con los outliers?

### Árbol de decisión

| Pregunta | Sí → | No → |
|---|---|---|
| ¿Es un error de medición o captura? | **Corregir o eliminar** | Seguir investigando |
| ¿Es el objetivo del análisis? (fraude, defectos) | **Mantener y estudiar** | Seguir investigando |
| ¿Distorsiona el análisis? | **Considerar tratamiento** | **Mantener sin cambios** |

### Técnicas de tratamiento

| Técnica | Qué hace | Cuándo usarla |
|---|---|---|
| **Eliminar** | Quita las filas con outliers | Error confirmado, pocas filas |
| **Winsorizar** | Recorta extremos al percentil 5/95 | Mantener la fila pero limitar impacto |
| **Capping (IQR)** | Reemplaza por el límite IQR | Similar a winsorizar |
| **Imputar** | Reemplaza por mediana | Si tratas el outlier como "dato faltante" |
| **Mantener** | No hacer nada | El outlier es información real y valiosa |

> **Siempre documenta qué hiciste y por qué.** "Eliminé 47 filas" sin explicación es mala práctica.

---

# PARTE 2: La Distribución Normal

## 2.1 ¿Qué es la distribución Normal?

La distribución Normal (o **Gaussiana**) es la distribución más importante de la estadística. Es la famosa "campana" — simétrica, con la mayoría de datos cerca del centro y pocos en los extremos.

$$X \sim N(\mu, \sigma^2)$$

Se lee: *"X sigue una distribución Normal con media μ y varianza σ²"*.

### ¿Por qué es tan común en la naturaleza?

Por el **Teorema del Límite Central**: cuando muchos factores pequeños e independientes se suman, el resultado tiende a ser Normal.

**Ejemplo:** La estatura de una persona depende de genética, nutrición, salud, ambiente, etc. Ningún factor domina → la suma de todos esos efectos tiende a una distribución Normal.

### Propiedades clave
- **Perfectamente simétrica**: media = mediana = moda, skewness = 0
- **Colas infinitas**: teóricamente va de −∞ a +∞, pero en la práctica casi todo está dentro de ±3σ
- **Estandarizable**: cualquier Normal se convierte en N(0, 1) con el Z-score

## 2.2 Los dos parámetros: μ y σ

La Normal se define **completamente** con solo dos números:

### μ (mu) — La Media
- Es el **centro** de la campana.
- Determina **dónde** se ubica la distribución en el eje X.
- Si μ cambia, la campana se **desplaza** a la izquierda o derecha, pero no cambia su forma.
- *Ejemplo: pH con μ = 3.31 está centrado en 3.31.*

### σ (sigma) — La Desviación Estándar
- Es el **ancho** de la campana.
- Determina qué tan **dispersos** están los datos alrededor de μ.
- σ pequeño → campana **alta y estrecha** (datos muy concentrados).
- σ grande → campana **baja y ancha** (datos muy dispersos).
- *Ejemplo: pH con σ = 0.15 es bastante concentrado.*

Con solo μ y σ puedes saber **todo** sobre una Normal: dónde está el centro, qué tan dispersa es, y qué porcentaje de datos cae en cualquier rango.

In [ ]:
# Simular una Normal en Python: np.random.normal(mu, sigma, n)
np.random.seed(42)
mu, sigma = 3.31, 0.15  # mismos parámetros que pH
datos_simulados = np.random.normal(mu, sigma, size=1599)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Normal simulada (perfecta)
sns.histplot(datos_simulados, kde=True, ax=axes[0], color="#2563EB", bins=30)
axes[0].set_title(f"Normal SIMULADA (μ={mu}, σ={sigma})")
axes[0].axvline(mu, color="#C82B40", ls="--", lw=2, label="μ")
axes[0].legend()

# pH real (casi normal)
sns.histplot(df["pH"], kde=True, ax=axes[1], color="#16A34A", bins=30)
axes[1].set_title("pH REAL del dataset")
axes[1].axvline(df["pH"].mean(), color="#C82B40", ls="--", lw=2, label="media")
axes[1].legend()

plt.suptitle("Normal simulada vs datos reales", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("La simulada es 'perfecta'. La real se le parece, pero no es idéntica.")
print("En la vida real, los datos NUNCA son perfectamente normales.")
print("La pregunta es: ¿se acercan lo suficiente?")

In [ ]:
# ¿Qué pasa cuando cambias μ y σ?
np.random.seed(42)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cambiar μ → el centro se mueve, la forma no cambia
for mu_val in [2.0, 3.31, 4.5]:
    data = np.random.normal(mu_val, 0.15, 5000)
    sns.kdeplot(data, ax=axes[0], label=f"μ={mu_val}", lw=2)
axes[0].set_title("Mismo σ, diferente μ\n(la campana se desplaza)", fontsize=11)
axes[0].legend()

# Cambiar σ → el ancho cambia
for sig_val in [0.05, 0.15, 0.40]:
    data = np.random.normal(3.31, sig_val, 5000)
    sns.kdeplot(data, ax=axes[1], label=f"σ={sig_val}", lw=2)
axes[1].set_title("Mismo μ, diferente σ\n(la campana se ensancha/estrecha)", fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

## 2.3 ¿Por qué importa saber si los datos son normales?

Muchas herramientas estadísticas **asumen** que los datos son normales. Si no lo verificas, tus conclusiones pueden ser **incorrectas**:

| Herramienta | Asume normalidad | Si no es Normal... |
|---|---|---|
| Test t de Student | Sí | p-value poco confiable |
| Intervalos de confianza | Sí (para la media) | Ancho incorrecto |
| Z-score para outliers | Sí | Detecta de más o de menos |
| Regla 68-95-99.7 | Sí | Los porcentajes no aplican |

### Ejemplos de conclusiones ERRADAS por asumir normalidad:

1. **"El 95% de los vinos tienen entre X e Y de azúcar"** — Usaste μ ± 2σ en `residual sugar` (que tiene skew = 1.9). La regla 68-95-99.7 NO aplica. El intervalo real es completamente diferente.

2. **"Solo el 0.3% de los chlorides son anómalos"** — Usaste Z-score > 3 en datos sesgados. En realidad hay un 5% de outliers, pero el Z-score no los detectó porque la media y std ya están infladas por los propios outliers.

3. **"La diferencia entre los grupos es estadísticamente significativa"** — Usaste un test t en datos con distribución sesgada. El p-value que obtuviste (0.03) en realidad debería ser 0.12. Conclusión opuesta.

> **Verificar normalidad no es un capricho académico. Es lo que te permite elegir la herramienta correcta.**

## 2.4 Tres herramientas para verificar normalidad

### Herramienta 1: Histograma + KDE (visual rápida)
¿Tiene forma de campana? ¿Es simétrica? ¿skew() ≈ 0?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# pH: candidata a normal
sns.histplot(df["pH"], kde=True, ax=axes[0], color="#16A34A", bins=30)
axes[0].set_title(f'pH (skew={df["pH"].skew():.2f})', fontsize=12)

# residual sugar: NO normal
sns.histplot(df["residual sugar"], kde=True, ax=axes[1], color="#C82B40", bins=30)
axes[1].set_title(f'residual sugar (skew={df["residual sugar"].skew():.2f})', fontsize=12)

plt.suptitle("¿Forma de campana?", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Herramienta 2: Q-Q Plot (visual precisa)

Compara los cuantiles de tus datos contra los cuantiles teóricos de una Normal perfecta.
- Si es Normal: los puntos caen **sobre la línea diagonal**.
- Si no: los puntos se desvían en las colas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# pH
stats.probplot(df["pH"], plot=axes[0])
axes[0].set_title("Q-Q Plot: pH", fontsize=12)
axes[0].get_lines()[0].set_color("#16A34A")
axes[0].get_lines()[1].set_color("#C82B40")

# residual sugar
stats.probplot(df["residual sugar"], plot=axes[1])
axes[1].set_title("Q-Q Plot: residual sugar", fontsize=12)
axes[1].get_lines()[0].set_color("#C82B40")
axes[1].get_lines()[1].set_color("#C82B40")

plt.tight_layout()
plt.show()

print("pH: los puntos siguen bastante bien la diagonal → aproximadamente Normal.")
print("residual sugar: los puntos se desvían fuertemente en la cola derecha → NO Normal.")

### Herramienta 3: Test de Shapiro-Wilk (numérico)

Un test estadístico formal:
- **p > 0.05**: no hay evidencia suficiente para rechazar normalidad ("podría ser Normal")
- **p ≤ 0.05**: se rechaza normalidad ("NO es Normal")

**Limitación:** con datasets grandes (n > 500), el test se vuelve muy sensible y rechaza casi todo. Por eso usamos una muestra.

In [ ]:
np.random.seed(42)

print(f"{'Variable':>22s}  {'Stat':>8s}  {'p-value':>10s}  Resultado")
print("-" * 62)

for col in ["pH", "density", "residual sugar", "chlorides", "alcohol"]:
    sample = df[col].sample(min(500, len(df)))
    stat, p = stats.shapiro(sample)
    resultado = "≈ Normal" if p > 0.05 else "NO Normal"
    print(f"{col:>22s}  {stat:>8.4f}  {p:>10.6f}  → {resultado}")

## 2.5 La regla 68-95-99.7

Si (y **solo si**) una variable sigue una distribución Normal:
- **68%** de los datos están dentro de μ ± 1σ
- **95%** dentro de μ ± 2σ
- **99.7%** dentro de μ ± 3σ

Verifiquemos con pH (≈ Normal) y con residual sugar (NO Normal):

In [ ]:
# pH (≈ Normal): la regla debería funcionar
col = "pH"
mu = df[col].mean()
sigma = df[col].std()

print(f"=== {col} (≈ Normal) ===")
print(f"μ = {mu:.2f}, σ = {sigma:.2f}\n")
for n_sigma, expected in [(1, 68), (2, 95), (3, 99.7)]:
    dentro = ((df[col] >= mu - n_sigma * sigma) &
              (df[col] <= mu + n_sigma * sigma)).mean() * 100
    print(f"  ±{n_sigma}σ [{mu - n_sigma*sigma:.2f}, {mu + n_sigma*sigma:.2f}]: "
          f"{dentro:.1f}% real vs {expected}% teórico")

In [ ]:
# residual sugar (NO Normal): la regla NO debería funcionar
col = "residual sugar"
mu = df[col].mean()
sigma = df[col].std()

print(f"=== {col} (NO Normal) ===")
print(f"μ = {mu:.2f}, σ = {sigma:.2f}\n")
for n_sigma, expected in [(1, 68), (2, 95), (3, 99.7)]:
    dentro = ((df[col] >= mu - n_sigma * sigma) &
              (df[col] <= mu + n_sigma * sigma)).mean() * 100
    print(f"  ±{n_sigma}σ [{mu - n_sigma*sigma:.2f}, {mu + n_sigma*sigma:.2f}]: "
          f"{dentro:.1f}% real vs {expected}% teórico")

print("\n⚠️ ¡Los porcentajes NO coinciden con la regla!")
print("Porque residual sugar NO es Normal. La regla no aplica aquí.")
print("Si hubieras reportado 'el 95% está entre X e Y', estarías EQUIVOCADO.")

## 2.6 ¿Y si no es Normal?

No todos los datos son normales, y **no necesitan serlo**. Lo importante es **saber** si lo son para elegir las herramientas correctas:

| Si es Normal... | Si NO es Normal... |
|---|---|
| Usa la **media** | Usa la **mediana** |
| Usa **std** | Usa **IQR** |
| Usa **Z-score** para outliers | Usa **método IQR** |
| Regla 68-95-99.7 **aplica** | **No aplica** |
| Test **paramétrico** (t, ANOVA) | Test **no paramétrico** (Mann-Whitney) |

> Verificar normalidad no es un capricho. Es lo que te permite **elegir la herramienta correcta** para cada situación.

---

# PARTE 3: Correlación

La correlación mide la **fuerza y dirección** de la relación **lineal** entre dos variables.

- **r = +1**: relación positiva perfecta (una sube → la otra sube)
- **r = 0**: sin relación **lineal** (¡puede haber otra!)
- **r = -1**: relación negativa perfecta (una sube → la otra baja)

> **Importante: r ≈ 0 NO significa "no hay relación". Significa que no hay relación LINEAL.**

## 3.1 Heatmap de correlación

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr(numeric_only=True)

# k=1 preserva la diagonal para que quality (última fila) sea visible
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title("Matriz de Correlación (Pearson) — Wine Quality", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ¿Qué variables correlacionan más con quality?
corr_quality = corr["quality"].drop("quality").sort_values(ascending=False)
print("Correlación con quality (Pearson):")
print(corr_quality.round(3))

## 3.2 Scatter plots para confirmar

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Correlación positiva más fuerte
sns.scatterplot(x="alcohol", y="quality", data=df,
                alpha=0.2, ax=axes[0], color="#16A34A")
r = df["alcohol"].corr(df["quality"])
axes[0].set_title(f"alcohol vs quality (r={r:.2f})", fontsize=11)

# Correlación negativa más fuerte
sns.scatterplot(x="volatile acidity", y="quality", data=df,
                alpha=0.2, ax=axes[1], color="#C82B40")
r = df["volatile acidity"].corr(df["quality"])
axes[1].set_title(f"volatile acidity vs quality (r={r:.2f})", fontsize=11)

# Sin correlación lineal
sns.scatterplot(x="residual sugar", y="quality", data=df,
                alpha=0.2, ax=axes[2], color="#6B7280")
r = df["residual sugar"].corr(df["quality"])
axes[2].set_title(f"residual sugar vs quality (r={r:.2f})", fontsize=11)

plt.tight_layout()
plt.show()

## 3.3 Correlación ≠ Causalidad

| | |
|---|---|
| **Mito** | "El alcohol causa que el vino sea bueno" (porque r ≈ 0.48) |
| **Realidad** | Los vinos con más alcohol suelen ser de uvas más maduras, que producen mejor sabor. La **madurez de la uva** es la variable oculta. |
| **Regla** | Correlación implica que **vale la pena investigar** si hay causalidad. No la prueba. |

> Ejemplo clásico: el consumo de helado y los ahogamientos correlacionan positivamente. ¿El helado causa ahogamientos? No — ambos aumentan en verano (variable oculta: temperatura).

## 3.4 Pearson vs Spearman

| | Pearson (r) | Spearman (ρ) |
|---|---|---|
| **Mide** | Relación **lineal** | Relación **monótona** (no necesita ser recta) |
| **Sensible a outliers** | Sí (usa valores) | No (usa rangos) |
| **Requiere normalidad** | Idealmente sí | No |
| **Cuándo usarlo** | Datos normales, sin outliers, relación lineal | Datos ordinales, outliers, relación curva pero consistente |

In [ ]:
# Comparar ambos métodos
cols = ["alcohol", "volatile acidity", "citric acid",
        "sulphates", "residual sugar", "chlorides"]

comparison = pd.DataFrame({
    "Pearson": [df[c].corr(df["quality"]) for c in cols],
    "Spearman": [df[c].corr(df["quality"], method="spearman") for c in cols],
}, index=cols)
comparison["Diferencia"] = abs(comparison["Pearson"] - comparison["Spearman"])
comparison = comparison.sort_values("Diferencia", ascending=False)

print("Pearson vs Spearman (correlación con quality):")
print(comparison.round(3))
print("\n→ Donde la diferencia es grande, la relación probablemente")
print("  NO es lineal o hay outliers influyentes.")

## 3.5 No todas las relaciones son lineales

**Pearson mide relación lineal.** Si la relación es curva, Pearson dirá "no hay relación" cuando **sí la hay**.

| Tipo de relación | Pearson | Spearman | ¿Qué hacer? |
|---|---|---|---|
| Lineal | Funciona bien | Funciona bien | Cualquiera |
| Monótona (curva pero consistente) | Subestima | **Funciona bien** | Usar Spearman |
| No monótona (U, olas) | Falla | Falla | Técnicas avanzadas |

> **Moraleja:** Siempre haz el scatter plot **antes** de confiar en el número r.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Relación lineal: alcohol vs quality
sns.regplot(x="alcohol", y="quality", data=df,
            scatter_kws={"alpha": 0.15, "color": "#16A34A"},
            line_kws={"color": "#C82B40"},
            ax=axes[0])
axes[0].set_title("alcohol vs quality (lineal)", fontsize=12)

# Relación no lineal: citric acid vs volatile acidity
sns.regplot(x="citric acid", y="volatile acidity", data=df,
            scatter_kws={"alpha": 0.15, "color": "#2563EB"},
            line_kws={"color": "#C82B40"},
            lowess=True,
            ax=axes[1])
axes[1].set_title("citric acid vs volatile acidity (curva suave - lowess)", fontsize=12)

plt.tight_layout()
plt.show()

print("lowess=True ajusta una curva suave en vez de una recta.")
print("Útil para explorar relaciones no lineales.")

---

# Ejercicio Integrador

Aplica los 3 temas al dataset Wine Quality.

### Parte A: Outliers

1. Detecta outliers en `total sulfur dioxide` usando el método IQR.
2. Visualiza con un boxplot.
3. ¿Qué calidad tienen los vinos con SO₂ extremo? ¿Son buenos o malos?

In [ ]:
# Tu código aquí


### Parte B: Normalidad

4. Verifica si `alcohol` es Normal (histograma, Q-Q plot, Shapiro-Wilk).
5. Simula 1599 datos normales con la misma media y std que `alcohol` y compáralos visualmente.

In [ ]:
# Tu código aquí


### Parte C: Correlación

6. Calcula la correlación (Pearson y Spearman) de `alcohol` con `quality`.
7. Haz un scatter plot con línea de regresión.
8. Identifica las 3 variables más correlacionadas con `quality` (positiva o negativamente).
9. **Bonus:** ¿Los outliers de SO₂ afectan la correlación? Calcula la correlación con y sin outliers.

In [ ]:
# Tu código aquí
